# New SQL Federation Features: GROUP BY, HAVING, ORDER BY, OFFSET

Demonstrates the new SQL-to-Cypher translation capabilities added in
[neo4j-jdbc PR #1310](https://github.com/neo4j/neo4j-jdbc/pull/1310).
These features enable `GROUP BY`, `HAVING`, `ORDER BY` on aggregates,
`OFFSET`, and `DISTINCT` to work through `remote_query()` via UC JDBC —
queries that previously required the Neo4j Spark Connector.

**What changed:** The JDBC translator now generates intermediate Cypher `WITH`
clauses to support explicit grouping keys and post-aggregation filtering.
This means most analytical queries can run through UC-governed `remote_query()`
without installing any cluster libraries.

## What's New

| SQL Feature | Status Before | Status Now | Cypher Strategy |
|-------------|---------------|------------|------------------|
| `GROUP BY` (projected columns) | Not supported | Supported | Implicit grouping via `RETURN` |
| `GROUP BY` (non-projected columns) | Not supported | Supported | `WITH` clause with `__group_col_N` |
| `HAVING` | Not supported | Supported | `WHERE` on `WITH` clause |
| `ORDER BY` on aggregate aliases | Not supported | Supported | `ORDER BY` after `RETURN` or `WITH` |
| `OFFSET` | Not supported | Supported | Cypher `SKIP` |
| `DISTINCT` + `GROUP BY` / `HAVING` | Not supported | Supported | `RETURN DISTINCT` |
| `JOIN` + `GROUP BY` | Not supported | Supported | Graph traversal + `WITH`/`RETURN` |
| `LIMIT` with `WITH` clauses | Not supported | Supported | `LIMIT` after `RETURN` |

All examples below use `remote_query()` — pure SQL, no Spark Connector needed.

## Prerequisites

1. **Neo4j UC JDBC connection** configured per [neo4j_uc_jdbc_guide.md](../docs/neo4j_uc_jdbc_guide.md)
2. **Neo4j JDBC driver** version 6.12.0+ (includes PR #1310 features)
3. **`neo4j-uc-creds` secret scope** configured via `setup.sh`
4. **Databricks preview features** enabled: custom JDBC on UC Compute, `remote_query`

---

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

SCOPE_NAME = "neo4j-uc-creds"

# Lakehouse configuration — update to match your environment
LAKEHOUSE_CATALOG = "aws-databricks-neo4j-lab"   # Your Unity Catalog name
LAKEHOUSE_SCHEMA = "lakehouse"                    # Schema containing Delta tables

# Neo4j credentials from Databricks Secrets
NEO4J_HOST = dbutils.secrets.get(SCOPE_NAME, "host")
NEO4J_USERNAME = dbutils.secrets.get(SCOPE_NAME, "user")
NEO4J_PASSWORD = dbutils.secrets.get(SCOPE_NAME, "password")
try:
    NEO4J_DATABASE = dbutils.secrets.get(SCOPE_NAME, "database")
except Exception:
    NEO4J_DATABASE = "neo4j"

UC_CONNECTION_NAME = dbutils.secrets.get(SCOPE_NAME, "connection_name")
NEO4J_BOLT_URI = f"neo4j+s://{NEO4J_HOST}"

# Set catalog and schema context for Delta table queries
spark.sql(f"USE CATALOG `{LAKEHOUSE_CATALOG}`")
spark.sql(f"USE SCHEMA `{LAKEHOUSE_SCHEMA}`")

print(f"Lakehouse: {LAKEHOUSE_CATALOG}.{LAKEHOUSE_SCHEMA}")
print(f"Neo4j Host: {NEO4J_HOST}")
print(f"Neo4j Bolt URI: {NEO4J_BOLT_URI}")
print(f"UC Connection: {UC_CONNECTION_NAME}")

---

## Section 1: GROUP BY via `remote_query()`

The translator handles two cases:

- **Implicit grouping:** When `GROUP BY` columns match projected columns, the
  translator emits a simple `MATCH ... RETURN` — Cypher's implicit grouping
  produces the correct result.
- **Explicit WITH clause:** When `GROUP BY` contains columns absent from the
  projection, the translator introduces a `WITH` clause with synthetic
  `__group_col_N` aliases.

**SQL → Cypher examples:**

| SQL | Cypher |
|-----|--------|
| `SELECT severity, COUNT(*) FROM MaintenanceEvent m GROUP BY severity` | `MATCH (m:MaintenanceEvent) RETURN m.severity AS severity, count(*)` |
| `SELECT COUNT(*) FROM MaintenanceEvent m GROUP BY severity` | `MATCH (m:MaintenanceEvent) WITH count(*) AS \`count(*)\`, m.severity AS __group_col_0 RETURN \`count(*)\`` |

In [ ]:
# GROUP BY: Implicit grouping — projected columns match GROUP BY columns
# SQL: SELECT severity, COUNT(*) AS cnt FROM MaintenanceEvent m GROUP BY severity
# Cypher: MATCH (m:MaintenanceEvent) RETURN m.severity AS severity, count(*) AS cnt

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT severity, COUNT(*) AS cnt FROM MaintenanceEvent m GROUP BY severity',
        outputSchema => 'severity STRING, cnt LONG')
""")

print("GROUP BY (implicit) — Maintenance events by severity")
print("Projected columns match GROUP BY → simple RETURN")
print("=" * 60)
result.show(truncate=False)

In [ ]:
# GROUP BY: Explicit WITH clause — GROUP BY column NOT in projection
# SQL: SELECT COUNT(*) AS cnt FROM MaintenanceEvent m GROUP BY severity
# Cypher: MATCH (m:MaintenanceEvent) WITH count(*) AS cnt, m.severity AS __group_col_0 RETURN cnt

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT COUNT(*) AS cnt FROM MaintenanceEvent m GROUP BY severity',
        outputSchema => 'cnt LONG')
""")

print("GROUP BY (explicit WITH) — Count per severity, severity not projected")
print("Non-projected GROUP BY column → WITH clause with __group_col_0")
print("=" * 60)
result.show(truncate=False)

In [ ]:
# GROUP BY: Multiple aggregates with implicit grouping
# SQL: SELECT operator, COUNT(*) AS flights, COUNT(DISTINCT origin) AS origins,
#      COUNT(DISTINCT destination) AS destinations FROM Flight f GROUP BY operator
# Cypher: MATCH (f:Flight) RETURN f.operator AS operator, count(*) AS flights,
#         count(DISTINCT f.origin) AS origins, count(DISTINCT f.destination) AS destinations

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT operator, COUNT(*) AS flights, COUNT(DISTINCT origin) AS origins, COUNT(DISTINCT destination) AS destinations FROM Flight f GROUP BY operator',
        outputSchema => 'operator STRING, flights LONG, origins LONG, destinations LONG')
""")

print("GROUP BY — Flight activity per operator with multiple aggregates")
print("=" * 60)
result.show(truncate=False)

---

## Section 2: HAVING Clause

The `HAVING` clause filters on aggregate results. The translator converts it to
a `WHERE` predicate on the `WITH` clause. Non-projected aggregates used only in
`HAVING` get synthetic `__having_col_N` aliases.

**SQL → Cypher examples:**

| SQL | Cypher |
|-----|--------|
| `SELECT name FROM People p GROUP BY name HAVING count(*) > 5` | `MATCH (p:People) WITH p.name AS name, count(*) AS __having_col_0 WHERE __having_col_0 > 5 RETURN name` |
| `SELECT name, count(*) AS cnt FROM People p GROUP BY name HAVING cnt > 5` | `MATCH (p:People) WITH p.name AS name, count(*) AS cnt WHERE cnt > 5 RETURN name, cnt` |

In [ ]:
# HAVING: Simple aggregate filter
# SQL: SELECT operator, COUNT(*) AS cnt FROM Flight f GROUP BY operator HAVING cnt > 20
# Cypher: MATCH (f:Flight) WITH f.operator AS operator, count(*) AS cnt WHERE cnt > 20 RETURN operator, cnt

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT operator, COUNT(*) AS cnt FROM Flight f GROUP BY operator HAVING cnt > 20',
        outputSchema => 'operator STRING, cnt LONG')
""")

print("HAVING — Operators with more than 20 flights")
print("=" * 60)
result.show(truncate=False)

In [ ]:
# HAVING: Non-projected aggregate (synthetic __having_col_0)
# SQL: SELECT severity FROM MaintenanceEvent m GROUP BY severity HAVING COUNT(*) > 10
# Cypher: MATCH (m:MaintenanceEvent) WITH m.severity AS severity,
#         count(*) AS __having_col_0 WHERE __having_col_0 > 10 RETURN severity

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT severity FROM MaintenanceEvent m GROUP BY severity HAVING COUNT(*) > 10',
        outputSchema => 'severity STRING')
""")

print("HAVING (non-projected aggregate) — Severities with >10 events")
print("COUNT(*) used in HAVING but not in SELECT → __having_col_0")
print("=" * 60)
result.show(truncate=False)

In [ ]:
# HAVING: Compound condition with multiple aggregates
# SQL: SELECT operator, COUNT(*) AS cnt FROM Flight f
#      GROUP BY operator HAVING COUNT(*) > 10 AND COUNT(DISTINCT origin) > 2
# Cypher: MATCH (f:Flight) WITH f.operator AS operator, count(*) AS cnt,
#         count(DISTINCT f.origin) AS __having_col_0
#         WHERE (cnt > 10 AND __having_col_0 > 2) RETURN operator, cnt

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT operator, COUNT(*) AS cnt FROM Flight f GROUP BY operator HAVING COUNT(*) > 10 AND COUNT(DISTINCT origin) > 2',
        outputSchema => 'operator STRING, cnt LONG')
""")

print("HAVING (compound) — Operators with >10 flights AND >2 unique origins")
print("=" * 60)
result.show(truncate=False)

---

## Section 3: ORDER BY on Aggregate Aliases

Results can now be ordered by aggregate aliases. The translator places `ORDER BY`
after the `RETURN` clause, or after the `WITH` clause when grouping requires it.

**SQL → Cypher example:**

| SQL | Cypher |
|-----|--------|
| `SELECT name, count(*) AS cnt FROM People p GROUP BY name ORDER BY cnt DESC` | `MATCH (p:People) RETURN p.name AS name, count(*) AS cnt ORDER BY cnt DESC` |

In [ ]:
# ORDER BY on aggregate alias
# SQL: SELECT severity, COUNT(*) AS cnt FROM MaintenanceEvent m
#      GROUP BY severity ORDER BY cnt DESC
# Cypher: MATCH (m:MaintenanceEvent) RETURN m.severity AS severity,
#         count(*) AS cnt ORDER BY cnt DESC

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT severity, COUNT(*) AS cnt FROM MaintenanceEvent m GROUP BY severity ORDER BY cnt DESC',
        outputSchema => 'severity STRING, cnt LONG')
""")

print("ORDER BY aggregate — Maintenance events ranked by count")
print("=" * 60)
result.show(truncate=False)

In [ ]:
# ORDER BY with multiple sort keys including aggregates
# SQL: SELECT operator, COUNT(*) AS cnt, COUNT(DISTINCT origin) AS routes
#      FROM Flight f GROUP BY operator ORDER BY cnt DESC, routes

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT operator, COUNT(*) AS cnt, COUNT(DISTINCT origin) AS routes FROM Flight f GROUP BY operator ORDER BY cnt DESC, routes',
        outputSchema => 'operator STRING, cnt LONG, routes LONG')
""")

print("ORDER BY (multi-key) — Operators sorted by flight count, then route count")
print("=" * 60)
result.show(truncate=False)

---

## Section 4: JOIN + GROUP BY (Graph Traversal with Aggregation)

Combines `NATURAL JOIN` (graph traversal) with `GROUP BY` to aggregate across
relationships. The translator converts SQL JOINs to Cypher pattern matches and
adds grouping via `RETURN` or `WITH` clauses.

**SQL → Cypher example:**

| SQL | Cypher |
|-----|--------|
| `SELECT c.name, count(*) FROM Customers c JOIN Orders o ON c.id = o.customer_id GROUP BY c.name` | `MATCH (c:Customers)<-[customer_id:CUSTOMER_ID]-(o:Orders) RETURN c.name, count(*)` |

In [ ]:
# JOIN + GROUP BY: Graph traversal with aggregation
# Counts flights per departure airport using Flight→DEPARTS_FROM→Airport traversal
#
# SQL:    SELECT COUNT(*) AS flight_count
#         FROM Flight f NATURAL JOIN DEPARTS_FROM r NATURAL JOIN Airport a
#         GROUP BY a.code
# Cypher: MATCH (f:Flight)-[:DEPARTS_FROM]->(a:Airport)
#         WITH count(*) AS flight_count, a.code AS __group_col_0 RETURN flight_count

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT COUNT(*) AS flight_count FROM Flight f NATURAL JOIN DEPARTS_FROM r NATURAL JOIN Airport a GROUP BY a.code',
        outputSchema => 'flight_count LONG')
""")

print("JOIN + GROUP BY — Flight counts grouped by departure airport")
print("Graph traversal: (Flight)-[:DEPARTS_FROM]->(Airport)")
print("=" * 60)
result.show(truncate=False)

In [ ]:
# JOIN + GROUP BY: Projected group key (implicit grouping)
# When the GROUP BY column is also projected, Cypher's implicit grouping works
#
# SQL:    SELECT a.code, COUNT(*) AS flight_count
#         FROM Flight f NATURAL JOIN DEPARTS_FROM r NATURAL JOIN Airport a
#         GROUP BY a.code
# Cypher: MATCH (f:Flight)-[:DEPARTS_FROM]->(a:Airport) RETURN a.code, count(*) AS flight_count

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT a.code, COUNT(*) AS flight_count FROM Flight f NATURAL JOIN DEPARTS_FROM r NATURAL JOIN Airport a GROUP BY a.code',
        outputSchema => 'code STRING, flight_count LONG')
""")

print("JOIN + GROUP BY (projected) — Departure airports with flight counts")
print("=" * 60)
result.show(truncate=False)

---

## Section 5: DISTINCT, LIMIT, and OFFSET

`DISTINCT` applies to the final `RETURN`, not the `WITH` clause.
`LIMIT` maps directly to Cypher `LIMIT` and `OFFSET` maps to `SKIP`.

**SQL → Cypher examples:**

| SQL | Cypher |
|-----|--------|
| `SELECT DISTINCT name, count(*) FROM People p GROUP BY name` | `MATCH (p:People) RETURN DISTINCT p.name AS name, count(*)` |
| `SELECT name FROM People p GROUP BY name HAVING count(*) > 5 ORDER BY name LIMIT 10 OFFSET 2` | `... RETURN name ORDER BY name SKIP 2 LIMIT 10` |

In [ ]:
# DISTINCT with GROUP BY
# SQL: SELECT DISTINCT operator, COUNT(*) AS cnt FROM Flight f GROUP BY operator
# Cypher: MATCH (f:Flight) RETURN DISTINCT f.operator AS operator, count(*) AS cnt

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT DISTINCT operator, COUNT(*) AS cnt FROM Flight f GROUP BY operator',
        outputSchema => 'operator STRING, cnt LONG')
""")

print("DISTINCT + GROUP BY — Unique operator flight counts")
print("=" * 60)
result.show(truncate=False)

In [ ]:
# LIMIT and OFFSET with GROUP BY
# SQL: SELECT operator, COUNT(*) AS cnt FROM Flight f
#      GROUP BY operator ORDER BY cnt DESC LIMIT 3 OFFSET 1
# Cypher: MATCH (f:Flight) RETURN f.operator AS operator,
#         count(*) AS cnt ORDER BY cnt DESC SKIP 1 LIMIT 3

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT operator, COUNT(*) AS cnt FROM Flight f GROUP BY operator ORDER BY cnt DESC LIMIT 3 OFFSET 1',
        outputSchema => 'operator STRING, cnt LONG')
""")

print("LIMIT + OFFSET — Page 2 of operators (skip 1, take 3)")
print("OFFSET → Cypher SKIP")
print("=" * 60)
result.show(truncate=False)

In [ ]:
# HAVING + ORDER BY + LIMIT + OFFSET combined
# SQL: SELECT severity, COUNT(*) AS cnt FROM MaintenanceEvent m
#      GROUP BY severity HAVING COUNT(*) > 5 ORDER BY cnt DESC LIMIT 10 OFFSET 0
# Cypher: MATCH (m:MaintenanceEvent) WITH m.severity AS severity, count(*) AS cnt
#         WHERE cnt > 5 RETURN severity, cnt ORDER BY cnt DESC SKIP 0 LIMIT 10

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT severity, COUNT(*) AS cnt FROM MaintenanceEvent m GROUP BY severity HAVING COUNT(*) > 5 ORDER BY cnt DESC LIMIT 10 OFFSET 0',
        outputSchema => 'severity STRING, cnt LONG')
""")

print("HAVING + ORDER BY + LIMIT + OFFSET — Filtered and paginated severity counts")
print("=" * 60)
result.show(truncate=False)

---

## Section 6: All Clauses Combined

A complex query exercising every new feature in a single `remote_query()` call:
`WHERE` + `GROUP BY` + `HAVING` + `ORDER BY` + `DISTINCT` + `LIMIT` + `OFFSET`.

**SQL → Cypher translation:**

```sql
SELECT DISTINCT severity, COUNT(*) AS cnt, MAX(fault) AS last_fault
FROM MaintenanceEvent m
WHERE aircraft_id LIKE 'AC%'
GROUP BY severity
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 10 OFFSET 0
```

Translates to:

```cypher
MATCH (m:MaintenanceEvent) WHERE m.aircraft_id STARTS WITH 'AC'
WITH m.severity AS severity, count(*) AS cnt, max(m.fault) AS last_fault
WHERE cnt > 1
RETURN DISTINCT severity, cnt, last_fault
ORDER BY cnt DESC SKIP 0 LIMIT 10
```

In [ ]:
# All clauses combined in a single remote_query()
# WHERE + GROUP BY + HAVING + ORDER BY + DISTINCT + LIMIT + OFFSET

result = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT DISTINCT severity, COUNT(*) AS cnt, MAX(fault) AS last_fault FROM MaintenanceEvent m WHERE aircraft_id LIKE ''AC%'' GROUP BY severity HAVING COUNT(*) > 1 ORDER BY cnt DESC LIMIT 10 OFFSET 0',
        outputSchema => 'severity STRING, cnt LONG, last_fault STRING')
""")

print("All clauses combined — WHERE + GROUP BY + HAVING + ORDER BY + DISTINCT + LIMIT + OFFSET")
print("=" * 80)
result.show(truncate=False)

---

## Section 7: Federated Query — Neo4j GROUP BY + Delta Lakehouse

The key benefit of GROUP BY support in `remote_query()`: Neo4j aggregate results
can now be JOINed with Delta lakehouse data **without the Spark Connector**.

Previously this required:
1. Loading all Neo4j rows via Spark Connector
2. Creating a temp view
3. Aggregating in Spark SQL

Now the aggregation happens in Neo4j and the grouped result set flows directly
through `remote_query()` into the federated query — fewer rows transferred,
no cluster library needed, fully UC-governed.

In [ ]:
# Federated Query: Neo4j GROUP BY via remote_query() + Delta sensor data
#
# Neo4j: Maintenance events grouped by aircraft_id and severity (via remote_query)
# Delta: Sensor health aggregates per aircraft
#
# Previously required Spark Connector — now pure SQL federation

result = spark.sql(f"""
    WITH neo4j_maint AS (
        SELECT *
        FROM remote_query('{UC_CONNECTION_NAME}',
            query => 'SELECT aircraft_id, severity, COUNT(*) AS cnt FROM MaintenanceEvent m GROUP BY aircraft_id, severity ORDER BY cnt DESC',
            outputSchema => 'aircraft_id STRING, severity STRING, cnt LONG')
    ),
    sensor_health AS (
        SELECT
            sys.aircraft_id,
            ROUND(AVG(CASE WHEN sen.type = 'EGT' THEN r.value END), 1) AS avg_egt,
            ROUND(AVG(CASE WHEN sen.type = 'Vibration' THEN r.value END), 4) AS avg_vibration
        FROM sensor_readings r
        JOIN sensors sen ON r.sensor_id = sen.`:ID(Sensor)`
        JOIN systems sys ON sen.system_id = sys.`:ID(System)`
        GROUP BY sys.aircraft_id
    )
    SELECT
        a.tail_number,
        a.model,
        m.severity,
        m.cnt AS maint_count,
        s.avg_egt AS avg_egt_c,
        s.avg_vibration AS avg_vib_ips
    FROM neo4j_maint m
    JOIN aircraft a ON m.aircraft_id = a.`:ID(Aircraft)`
    JOIN sensor_health s ON m.aircraft_id = s.aircraft_id
    ORDER BY m.cnt DESC
""")

print("Federated: Neo4j GROUP BY (remote_query) + Delta sensor health")
print("Neo4j: MaintenanceEvent grouped by aircraft/severity via UC JDBC")
print("Delta: sensor_readings, sensors, systems, aircraft")
print("=" * 80)
result.show(30, truncate=False)

In [ ]:
# Federated Query: Neo4j HAVING filter + Delta lakehouse
#
# Neo4j: Only operators with >20 flights (filtered via HAVING in Neo4j)
# Delta: Engine performance metrics per aircraft
#
# The HAVING filter runs in Neo4j — only qualifying rows cross the federation boundary

result = spark.sql(f"""
    WITH active_operators AS (
        SELECT *
        FROM remote_query('{UC_CONNECTION_NAME}',
            query => 'SELECT operator, COUNT(*) AS flight_count, COUNT(DISTINCT aircraft_id) AS aircraft_count FROM Flight f GROUP BY operator HAVING COUNT(*) > 20 ORDER BY flight_count DESC',
            outputSchema => 'operator STRING, flight_count LONG, aircraft_count LONG')
    ),
    fleet_sensor_avg AS (
        SELECT
            a.operator,
            ROUND(AVG(CASE WHEN sen.type = 'EGT' THEN r.value END), 1) AS avg_egt,
            ROUND(AVG(CASE WHEN sen.type = 'FuelFlow' THEN r.value END), 2) AS avg_fuel_flow,
            ROUND(AVG(CASE WHEN sen.type = 'N1Speed' THEN r.value END), 0) AS avg_n1_speed
        FROM sensor_readings r
        JOIN sensors sen ON r.sensor_id = sen.`:ID(Sensor)`
        JOIN systems sys ON sen.system_id = sys.`:ID(System)`
        JOIN aircraft a ON sys.aircraft_id = a.`:ID(Aircraft)`
        WHERE sys.type = 'Engine'
        GROUP BY a.operator
    )
    SELECT
        o.operator,
        o.flight_count,
        o.aircraft_count,
        s.avg_egt AS avg_egt_c,
        s.avg_fuel_flow AS fuel_kgs,
        s.avg_n1_speed AS n1_rpm
    FROM active_operators o
    JOIN fleet_sensor_avg s ON o.operator = s.operator
    ORDER BY o.flight_count DESC
""")

print("Federated: Neo4j HAVING filter + Delta engine performance")
print("Neo4j: Operators with >20 flights (HAVING filters in Neo4j)")
print("Delta: Engine sensor averages per operator")
print("=" * 80)
result.show(truncate=False)

---

## Summary

This notebook demonstrated the new SQL-to-Cypher translation capabilities from
[neo4j-jdbc PR #1310](https://github.com/neo4j/neo4j-jdbc/pull/1310), all running
through `remote_query()` via UC JDBC.

| Section | Feature | What It Shows |
|---------|---------|---------------|
| 1 | `GROUP BY` | Implicit grouping (RETURN) and explicit WITH clause generation |
| 2 | `HAVING` | Simple, alias-based, non-projected, and compound conditions |
| 3 | `ORDER BY` | Sorting on aggregate aliases, multi-key ordering |
| 4 | `JOIN` + `GROUP BY` | Graph traversal with aggregation |
| 5 | `DISTINCT`, `LIMIT`, `OFFSET` | Pagination and deduplication with aggregates |
| 6 | All combined | WHERE + GROUP BY + HAVING + ORDER BY + DISTINCT + LIMIT + OFFSET |
| 7 | Federated queries | Neo4j GROUP BY/HAVING via remote_query() + Delta lakehouse |

### Impact on Federation Architecture

| Capability | Before (02_federated_query_patterns) | Now |
|-----------|--------------------------------------|-----|
| Aggregate queries | `remote_query()` | `remote_query()` |
| GROUP BY / HAVING | Spark Connector only | `remote_query()` |
| ORDER BY on aggregates | Spark Connector only | `remote_query()` |
| Pagination (OFFSET) | Spark Connector only | `remote_query()` |
| Row-level data | Spark Connector | Spark Connector |

The Spark Connector is still needed for non-aggregate `SELECT` queries and
relationship property aggregation. But for analytical grouping and filtering,
`remote_query()` now handles the most common patterns — with full UC governance,
no cluster library required.

### References

- [neo4j-jdbc PR #1310](https://github.com/neo4j/neo4j-jdbc/pull/1310) — Implementation details
- [Neo4j JDBC SQL2Cypher](https://neo4j.com/docs/jdbc-manual/current/sql2cypher/) — Translation rules
- [02_federated_query_patterns](./02_federated_query_patterns.ipynb) — Original federation patterns
- [Databricks remote_query()](https://docs.databricks.com/sql/language-manual/functions/remote_query) — Table-valued function reference